In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

from orbit_functions import gauss_uv, propagate_orbit, R_EARTH, MU_EARTH
from orbit_plotting import plot_gauss

In [ ]:
# test scenario
r1 = np.array([R_EARTH + 600e3, 0, 0])
r2 = np.array([-R_EARTH - 400e3, R_EARTH, -0.5*R_EARTH])
dt = 60*60
(sol_short, sol_long) = gauss_uv(r1, r2, dt)

fig1 = plot_gauss(sol_short)
fig2 = plot_gauss(sol_long)

fig1.show()
fig2.show()

In [ ]:
# target
radius_target_orbit = R_EARTH + 750e3
r_target0 = radius_target_orbit * np.array([1, 1, 0]) / math.sqrt(2)
v_target0 = math.sqrt(MU_EARTH/radius_target_orbit) * np.array([-1, 1, 0]) / math.sqrt(2)

# intercepter
radius_intercepter = R_EARTH + 300e3
r_intercepter0 = radius_intercepter * np.array([1, 2, 0]) / math.sqrt(5)
v_intercepter0 = math.sqrt(MU_EARTH/radius_intercepter) * np.array([-2, 1, 1]) / math.sqrt(6)


period_target = 2 * np.pi * np.sqrt(radius_target_orbit**3 / MU_EARTH)
intercept_times = np.linspace(0.1*period_target, 0.9*period_target, 200)

r_target = np.zeros((3, np.size(intercept_times)))
r_intercepter = np.zeros((3, np.size(intercept_times)))

v_target = np.zeros((3, np.size(intercept_times)))
v_intercepter = np.zeros((3, np.size(intercept_times)))

for i in range(np.size(intercept_times)):
    r1, v1 = propagate_orbit(r_target0, v_target0, intercept_times[i], MU_EARTH)
    r_target[:,i] = r1
    v_target[:,i] = v1

    r1, v1 = propagate_orbit(r_intercepter0, v_intercepter0, intercept_times[i], MU_EARTH)
    r_intercepter[:,i] = r1
    v_intercepter[:,i] = v1

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# plot earth
u, v = np.mgrid[0:2*np.pi:30j, 0:np.pi:20j]
x_earth = R_EARTH * np.cos(u) * np.sin(v)
y_earth = R_EARTH * np.sin(u) * np.sin(v)
z_earth = R_EARTH * np.cos(v)
ax.plot_surface(x_earth, y_earth, z_earth, color="blue", alpha=0.5)

# plot orbit
ax.plot(r_target[0,:], r_target[1,:], r_target[2,:], "b-", label="Orbit")
ax.scatter(r_target[0, 0], r_target[1, 0], r_target[2, 0], c="b", marker="o", label="Start")
ax.scatter(r_target[0, -1], r_target[1,-1], r_target[2,-1], c="b", marker="x", label="End")

ax.plot(r_intercepter[0,:], r_intercepter[1,:], r_intercepter[2,:], "r-", label="Orbit")
ax.scatter(r_intercepter[0, 0], r_intercepter[1, 0], r_intercepter[2, 0], c="r", marker="o", label="Start")
ax.scatter(r_intercepter[0, -1], r_intercepter[1,-1], r_intercepter[2,-1], c="r", marker="x", label="End")

ax.set_xlabel("X (km)")
ax.set_ylabel("Y (km)")
ax.set_zlabel("Z (km)")
ax.set_title("Target and intercepter original orbits")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# find cheapest dV

dVs = np.zeros_like(intercept_times)


for i in range(np.size(intercept_times)):
    dt = intercept_times[i]

    # gauss solver to get trajectories assuming we start maneuver at t=0
    (sol_short, sol_long) = gauss_uv(r_intercepter[:,0], r_target[:,i], dt)

    dV1_short = np.linalg.norm(sol_short.v1 - v_intercepter[:,0])
    dV2_short = np.linalg.norm(sol_short.v2 - v_target[:,i])

    dV1_long = np.linalg.norm(sol_long.v1 - v_intercepter[:,0])
    dV2_long = np.linalg.norm(sol_long.v2 - v_target[:,i])

    dVs[i] = min(dV1_short + dV2_short, dV1_long + dV2_long)


plt.figure()

plt.plot(intercept_times, dVs)

plt.ylim(0, np.max(dVs))

plt.grid()
plt.xlabel("Intercept time [s]")
plt.ylabel("dV [m/s]")
plt.show()



In [ ]:
# r_target = np.zeros((3, np.size(intercept_times)))
# r_intercepter = np.zeros((3, np.size(intercept_times)))

# v_target = np.zeros((3, np.size(intercept_times)))
# v_intercepter = np.zeros((3, np.size(intercept_times)))

# for i in range(np.size(intercept_times)):
#     r1, v1 = propagate_orbit(r_target0, v_target0, intercept_times[i], MU_EARTH)
#     r_target[:,i] = r1
#     v_target[:,i] = v1

#     r1, v1 = propagate_orbit(r_intercepter0, v_intercepter0, intercept_times[i], MU_EARTH)
#     r_intercepter[:,i] = r1
#     v_intercepter[:,i] = v1



period_target = 2 * np.pi * np.sqrt(radius_target_orbit**3 / MU_EARTH)
m1_t = np.linspace(0.0*period_target, 10*period_target, 200)

transfer_times = np.linspace(0.1*period_target, 0.9*period_target, 200)


dVs = np.zeros((m1_t.size, transfer_times.size))

for i in range(m1_t.size):
    for j in range(transfer_times.size):
        dt = transfer_times[j]
        m2_t = m1_t[i] + dt

        # get position of intercepter at m1_t 
        r1_intercepter, v1_intercepter = propagate_orbit(r_intercepter0, v_intercepter0, m1_t[i], MU_EARTH)

        # get position of target at m2_t
        r1_targ, v1_targ = propagate_orbit(r_target0, v_target0, m2_t, MU_EARTH)


        # gauss solver to get trajectories assuming we start maneuver at t=0
        (sol_short, sol_long) = gauss_uv(r1_intercepter, r1_targ, dt)

        dV1_short = np.linalg.norm(sol_short.v1 - v1_intercepter)
        dV2_short = np.linalg.norm(sol_short.v2 - v1_targ)

        dV1_long = np.linalg.norm(sol_long.v1 - v1_intercepter)
        dV2_long = np.linalg.norm(sol_long.v2 - v1_targ)

        dVs[i,j] = min(dV1_short + dV2_short, dV1_long + dV2_long)




In [ ]:
fig = plt.figure(figsize=(12,6))

ax = fig.add_subplot(projection="3d")

X, Y = np.meshgrid(m1_t, transfer_times, indexing="ij")
surf = ax.plot_surface(
    X, Y, dVs,
    cmap = "viridis"
)


ax.view_init(elev=30, azim=30)

fig.colorbar(surf, shrink=0.5, aspect=10)

ax.set_xlabel("M1 time [s]")
ax.set_ylabel("Transfer time [s]")
# ax.set_zlabel("dV [m/s]")

plt.tight_layout()
plt.show()

In [ ]:
r0 = np.array([2, 1, 0]) / math.sqrt(5)
v0 = np.array([-1, 2, 1]) / math.sqrt(6)

print("r0 and v0")
print(r0)
print(v0)

print(r0.dot(v0))